# BrandSphere AI — Week 6: Animated Visuals Studio
**CRS AI Capstone 2025-26 | Scenario 1**

This notebook:
1. Imports logo, font, and color palette assets
2. Plans animation style (typewriter + slide-in)
3. Creates animated GIF using Pillow
4. Exports at social media resolution (600x340)
5. Integrates with Streamlit preview

In [ ]:
!pip install pillow matplotlib numpy -q
print('✅ Dependencies installed')

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import io, os, json, math

# Load assets from previous weeks
try:
    with open('generated_slogans.json') as f:
        slogans_data = json.load(f)
    tagline = slogans_data.get('top_slogan', 'Built for clarity.')
except:
    tagline = 'Built for clarity.'

palette = ['#1B3A6B', '#C9A84C', '#3ECFB2', '#F0EDE8', '#FFFFFF']
company = 'TechNova'

print(f'✅ Assets loaded')
print(f'   Tagline: {tagline}')
print(f'   Palette: {palette}')
print(f'   Company: {company}')

In [ ]:
# Animation storyboard
print('🎬 Animation Storyboard:')
print('  Frame  1-4:  Logo slides in from left')
print('  Frame  3-6:  Vertical divider line grows')
print('  Frame  5-10: Tagline typewriter effect')
print('  Frame  8-12: Brand colors animate in')
print('  Frame 10-12: AI label fades in')
print('  Total: 24 frames @ 80ms = ~2 second loop')

In [ ]:
def h2r(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def create_logo_placeholder(company, palette):
    """Create simple logo for animation demo."""
    W = H = 120
    c1 = palette[0]; c2 = palette[1]; c3 = palette[4]
    r1,g1,b1 = h2r(c1); r2,g2,b2 = h2r(c2)
    img = Image.new('RGB', (W,H), (r1,g1,b1))
    draw = ImageDraw.Draw(img)
    for rad in range(55,0,-3):
        a=(55-rad)/55
        draw.ellipse([W//2-rad,H//2-rad,W//2+rad,H//2+rad],
                     fill=(min(255,int(r1+(255-r1)*a*0.25)),
                           min(255,int(g1+(255-g1)*a*0.25)),
                           min(255,int(b1+(255-b1)*a*0.25))))
    initials = ''.join([w[0].upper() for w in company.split()[:2]])
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 36)
    except:
        font = ImageFont.load_default()
    bb = draw.textbbox((0,0), initials, font=font)
    draw.text((W//2-(bb[2]-bb[0])//2, H//2-(bb[3]-bb[1])//2), initials, fill=(r2,g2,b2), font=font)
    return img

logo_img = create_logo_placeholder(company, palette)
logo_img.save('demo_logo.png')
print('✅ Demo logo created')

In [ ]:
def create_animated_gif(logo_img, tagline, palette, company, n_frames=24):
    """Create professional animated brand GIF."""
    W, H = 600, 340
    c1, c2 = palette[0], palette[1]
    r1,g1,b1 = h2r(c1); r2,g2,b2 = h2r(c2)

    try:
        ft = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 24)
        fs = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 13)
        fc = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 16)
    except:
        ft = fs = fc = ImageFont.load_default()

    logo_resized = logo_img.resize((110,110)).convert('RGBA')
    tag = tagline[:52]+('...' if len(tagline)>52 else '')
    frames = []

    for i in range(n_frames):
        t = i/(n_frames-1)
        frame = Image.new('RGB', (W,H), (r1,g1,b1))
        draw = ImageDraw.Draw(frame)

        # Gradient background
        for x in range(W):
            a = x/W*0.10
            draw.line([(x,0),(x,H)], fill=(min(255,int(r1+(255-r1)*a)),
                                           min(255,int(g1+(255-g1)*a)),
                                           min(255,int(b1+(255-b1)*a))))
        # Accent bar
        draw.rectangle([0,H-5,int(W*min(1.0,t*1.4)),H], fill=(r2,g2,b2))

        # Logo slide in
        lx = int(-110+160*min(1.0,t*2.2))
        if lx > -110:
            frame.paste(logo_resized, (max(0,lx), H//2-55), logo_resized)

        # Divider
        if t > 0.3:
            la = min(1.0,(t-0.3)/0.25)
            lh = int(180*la)
            draw.line([(185,H//2-lh//2),(185,H//2+lh//2)], fill=(r2,g2,b2), width=2)

        # Company name
        if t > 0.25:
            ca = min(1.0,(t-0.25)/0.3)
            cox = int(205+(1-ca)*35)
            draw.text((cox,H//2-62), company.upper()[:20], fill=(r2,g2,b2), font=fs)

        # Typewriter tagline
        if t > 0.38:
            chars = int(len(tag)*min(1.0,(t-0.38)/0.45))
            txt = tag[:chars]+('|' if i%4<2 and chars<len(tag) else '')
            draw.text((205,H//2-18), txt, fill=(255,255,255), font=ft)

        # Sub label
        if t > 0.78:
            fa = min(1.0,(t-0.78)/0.18)
            gc2 = int(110*fa)
            draw.text((205,H//2+22), '— AI Generated by BrandSphere', fill=(gc2,gc2,gc2), font=fs)

        # Animated dots
        for di in range(3):
            dt = (t*2.5+di*0.33)%1.0
            dsz = int(3+3*math.sin(dt*math.pi))
            dx = W-38+di*11; dy=18
            bri = int(80+175*dt)
            draw.ellipse([dx,dy,dx+dsz,dy+dsz], fill=(bri,bri,bri))

        frames.append(frame)

    buf = io.BytesIO()
    frames[0].save(buf, format='GIF', save_all=True,
                   append_images=frames[1:], duration=80, loop=0, optimize=True)
    return buf.getvalue()

gif_bytes = create_animated_gif(logo_img, tagline, palette, company)
with open('brand_animation.gif', 'wb') as f:
    f.write(gif_bytes)
print(f'✅ Animated GIF saved: {len(gif_bytes)/1024:.1f} KB')

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage('brand_animation.gif'))
print('✅ Animation displayed above')

In [ ]:
import shutil
from google.colab import drive
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/BrandSphere_Models'
os.makedirs(save_dir, exist_ok=True)
shutil.copy('brand_animation.gif', f'{save_dir}/brand_animation.gif')
shutil.copy('demo_logo.png', f'{save_dir}/demo_logo.png')
print('✅ Animation uploaded to Drive')

print('=' * 55)
print('  WEEK 6 DELIVERABLES COMPLETE')
print('=' * 55)
print('  ✅ Logo asset prepared')
print('  ✅ Animation storyboard planned')
print('  ✅ 24-frame GIF created')
print('  ✅ Typewriter + slide-in effects')
print('  ✅ Brand colors applied')
print('  ✅ GIF exported at 600x340 (social media ready)')
print('  ✅ Integrated with Streamlit Content Hub tab')
print('=' * 55)